In [ ]:
!pip install groq
!pip install langgraph -q
!pip install langchain-groq
!pip install langchain langchain-groq -q
!pip install --upgrade langchain langchain-groq langchain-community langchain-core -q

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import os
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

os.environ['GROQ_API_KEY'] = 'your-key-here'

model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── Make sure all tools have defensive float conversion ───────

@tool
def search_properties(area: str) -> str:
    """Search for property listings in a specific London area.
    Returns average price, listings count, yield and property types."""
    data = {
        "hackney":       {"avg_price": 485000, "listings": 12, "yield": 4.2, "type": "mixed"},
        "croydon":       {"avg_price": 220000, "listings": 28, "yield": 6.1, "type": "flats"},
        "surrey":        {"avg_price": 875000, "listings": 5,  "yield": 2.9, "type": "houses"},
        "bethnal green": {"avg_price": 395000, "listings": 8,  "yield": 4.9, "type": "flats"},
        "canary wharf":  {"avg_price": 550000, "listings": 15, "yield": 3.8, "type": "flats"},
    }
    area_lower = area.lower()
    if area_lower in data:
        d = data[area_lower]
        return (f"Area: {area}. Average price: £{d['avg_price']:,}. "
                f"Listings: {d['listings']}. Rental yield: {d['yield']}%. "
                f"Property types: {d['type']}.")
    return f"No data found for {area}. Available: Hackney, Croydon, Surrey, Bethnal Green, Canary Wharf."

@tool
def calculate_yield(price: float, monthly_rent: float) -> str:
    """Calculate annual rental yield for a property.
    Args: price (number only e.g. 350000), monthly_rent (number only e.g. 1500)"""
    price        = float(str(price).replace('£','').replace(',','').strip())
    monthly_rent = float(str(monthly_rent).replace('£','').replace(',','').strip())
    annual_rent  = monthly_rent * 12
    yield_pct    = (annual_rent / price) * 100
    return (f"Purchase: £{price:,.0f}. Monthly rent: £{monthly_rent:,.0f}. "
            f"Annual rent: £{annual_rent:,.0f}. Yield: {yield_pct:.2f}%.")

@tool
def compare_yields(areas: str) -> str:
    """Compare rental yields across multiple London areas.
    Args: areas (comma-separated area names e.g. Hackney, Croydon)"""
    yields = {
        "hackney": 4.2, "croydon": 6.1, "surrey": 2.9,
        "bethnal green": 4.9, "canary wharf": 3.8
    }
    area_list = [a.strip().lower() for a in areas.split(",")]
    results   = [(a.title(), yields[a]) for a in area_list if a in yields]
    if not results:
        return "No yield data found."
    ranked = sorted(results, key=lambda x: x[1], reverse=True)
    return "Yields ranked: " + " | ".join([f"{a}: {y}%" for a,y in ranked])

@tool
def get_market_trend(area: str) -> str:
    """Get price growth forecast for a London area in 2025.
    Args: area (area name e.g. Hackney)"""
    trends = {
        "hackney":       {"growth": 4.5, "outlook": "strong demand, limited supply"},
        "croydon":       {"growth": 6.2, "outlook": "regeneration driving growth"},
        "surrey":        {"growth": 3.1, "outlook": "stable, family home demand"},
        "bethnal green": {"growth": 5.0, "outlook": "gentrification continuing"},
        "canary wharf":  {"growth": 2.8, "outlook": "oversupply of new builds"},
    }
    area_lower = area.lower()
    if area_lower in trends:
        t = trends[area_lower]
        return (f"{area} 2025 forecast: {t['growth']}% price growth. "
                f"Outlook: {t['outlook']}.")
    return f"No trend data for {area}."

@tool
def get_mortgage_cost(purchase_price: float, deposit_pct: float) -> str:
    """Calculate monthly mortgage payment.
    Args: purchase_price (number only e.g. 300000), deposit_pct (number only e.g. 25)"""
    purchase_price = float(str(purchase_price).replace('£','').replace(',','').strip())
    deposit_pct    = float(str(deposit_pct).replace('%','').strip())
    deposit        = purchase_price * (deposit_pct / 100)
    loan           = purchase_price - deposit
    monthly_rate   = 5.5 / 100 / 12
    months         = 25 * 12
    payment        = loan * (monthly_rate * (1+monthly_rate)**months) / \
                     ((1+monthly_rate)**months - 1)
    return (f"Purchase: £{purchase_price:,.0f}. "
            f"Deposit ({deposit_pct}%): £{deposit:,.0f}. "
            f"Loan: £{loan:,.0f}. "
            f"Monthly mortgage: £{payment:,.0f} at 5.5% over 25 years.")

# ── Rebuild agent with all tools ─────────────────────────────
tools         = [search_properties, calculate_yield,
                 compare_yields, get_market_trend, get_mortgage_cost]
advisor_agent = create_react_agent(model, tools)
print(f"✓ Agent ready with {len(tools)} tools")

# ── FIXED system prompt — no tool/search/call words ──────────
ADVISOR_SYSTEM_PROMPT = """You are Alex, a senior London property
investment advisor with 20 years of experience.

Your expertise:
- Rental yield analysis and investment returns
- Mortgage affordability and calculations
- London area trends and price forecasts
- First time buyer and investor guidance

Your rules:
- Always provide specific figures in your answers
- Always mention risks alongside opportunities
- Always compare at least two areas when recommending
- End every investment recommendation with a risk disclaimer
- Be honest about areas outside your knowledge

Your personality:
- Professional, warm, and trustworthy
- Specific — cite exact numbers wherever possible
- Balanced — never oversell any investment
"""

# ── Conversation with memory ──────────────────────────────────
advisor_history = []

def ask_alex(question):
    messages = (
        [{"role": "system", "content": ADVISOR_SYSTEM_PROMPT}]
        + [{"role": "user" if isinstance(m, HumanMessage) else "assistant",
            "content": m.content} for m in advisor_history]
        + [{"role": "user", "content": question}]
    )

    result = advisor_agent.invoke({"messages": messages})
    answer = result["messages"][-1].content

    advisor_history.append(HumanMessage(content=question))
    advisor_history.append(AIMessage(content=answer))

    print(f"\nYou : {question}")
    print(f"Alex: {answer}")
    print(f"[{len(advisor_history)//2} turns remembered]\n")

# ── Run all four turns ────────────────────────────────────────
print("="*55)
print("PropertyAI — Your London Investment Advisor")
print("="*55)

ask_alex("Hi, I'm looking to invest in London property. I have £90,000 saved for a deposit.")
ask_alex("Which areas would you recommend for the best yield?")
ask_alex("Tell me more about Croydon — what would my mortgage look like?")
ask_alex("What are the main risks I should be aware of before investing?")